In [24]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [25]:
# Device
# =========================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using:", device)


# Data
# =========================================================

transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False
)

# Training
# =========================================================

def train_epoch():

    model.train()

    total_loss = 0

    for x, y in train_loader:

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        out = model(x)

        loss = F.cross_entropy(out, y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)


# Evaluation
# =========================================================

def evaluate():

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for x, y in test_loader:

            x = x.to(device)
            y = y.to(device)

            out = model(x)

            preds = out.argmax(dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total

Using: cpu


In [26]:
# Vanilla Sigmoid Network
# =========================================================

class VanillaSigmoidNet(nn.Module):

    def __init__(self,
                 hidden1=256,
                 hidden2=128):

        super().__init__()

        self.act = nn.Sigmoid()

        self.fc1 = nn.Linear(28 * 28, hidden1)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.fc3 = nn.Linear(hidden2, 10)

    def forward(self, x):

        x = x.view(x.size(0), -1)

        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))

        x = self.fc3(x)

        return x


# =========================================================
# Model
# =========================================================

model = VanillaSigmoidNet().to(device)

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3
)

# =========================================================
# Run
# =========================================================

EPOCHS = 5

for epoch in range(EPOCHS):

    loss = train_epoch()

    acc = evaluate()

    print(
        f"Epoch {epoch+1} | "
        f"Loss = {loss:.4f} | "
        f"Test Accuracy = {100*acc:.2f}%"
    )

Epoch 1 | Loss = 0.7046 | Test Accuracy = 91.88%
Epoch 2 | Loss = 0.2300 | Test Accuracy = 94.37%
Epoch 3 | Loss = 0.1666 | Test Accuracy = 95.47%
Epoch 4 | Loss = 0.1292 | Test Accuracy = 96.26%
Epoch 5 | Loss = 0.1025 | Test Accuracy = 96.68%


In [40]:
# GELU Network
# =========================================================

class GELUNet(nn.Module):

    def __init__(self,
                 hidden1=256,
                 hidden2=128):

        super().__init__()

        self.act = nn.GELU()

        self.fc1 = nn.Linear(28 * 28, hidden1)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.fc3 = nn.Linear(hidden2, 10)

    def forward(self, x):

        x = x.view(x.size(0), -1)

        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))

        x = self.fc3(x)

        return x


# =========================================================
# Model
# =========================================================

model = GELUNet().to(device)

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3
)


# =========================================================
# Run
# =========================================================

EPOCHS = 5

for epoch in range(EPOCHS):

    loss = train_epoch()

    acc = evaluate()

    print(
        f"Epoch {epoch+1} | "
        f"Loss = {loss:.4f} | "
        f"Test Accuracy = {100*acc:.2f}%"
    )

Epoch 1 | Loss = 0.3486 | Test Accuracy = 95.05%
Epoch 2 | Loss = 0.1392 | Test Accuracy = 96.44%
Epoch 3 | Loss = 0.0916 | Test Accuracy = 97.00%
Epoch 4 | Loss = 0.0666 | Test Accuracy = 97.66%
Epoch 5 | Loss = 0.0494 | Test Accuracy = 97.74%


In [41]:
# Swish / SiLU Network
# =========================================================

class SwishNet(nn.Module):

    def __init__(self,
                 hidden1=256,
                 hidden2=128):

        super().__init__()

        self.act = nn.SiLU()

        self.fc1 = nn.Linear(28 * 28, hidden1)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.fc3 = nn.Linear(hidden2, 10)

    def forward(self, x):

        x = x.view(x.size(0), -1)

        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))

        x = self.fc3(x)

        return x


# =========================================================
# Model
# =========================================================

model = SwishNet().to(device)

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3
)


# =========================================================
# Run
# =========================================================

EPOCHS = 5

for epoch in range(EPOCHS):

    loss = train_epoch()

    acc = evaluate()

    print(
        f"Epoch {epoch+1} | "
        f"Loss = {loss:.4f} | "
        f"Test Accuracy = {100*acc:.2f}%"
    )

Epoch 1 | Loss = 0.3627 | Test Accuracy = 94.98%
Epoch 2 | Loss = 0.1426 | Test Accuracy = 96.39%
Epoch 3 | Loss = 0.0985 | Test Accuracy = 97.22%
Epoch 4 | Loss = 0.0731 | Test Accuracy = 97.61%
Epoch 5 | Loss = 0.0548 | Test Accuracy = 97.31%


In [28]:
# Sinunet definition

# Fast Sinusoidal CDF Activation
# =========================================================

class FastSinCDFActivation(nn.Module):

    def __init__(self, s=1.0, k=2.0, resolution=5000):
        super().__init__()

        self.s = s
        self.k = k
        self.resolution = resolution

        # -------------------------------------------------
        # Precompute lookup table
        # -------------------------------------------------

        xgrid = torch.linspace(0, 1, resolution)

        eps = 1e-6

        # PDF
        pdf = (torch.sin(torch.pi * (xgrid ** s)) + eps) ** k

        # Normalize
        A = torch.trapz(pdf, xgrid)
        pdf = pdf / A

        # Approximate CDF
        dx = xgrid[1] - xgrid[0]
        cdf = torch.cumsum(pdf, dim=0) * dx

        # Ensure exact endpoint
        cdf[-1] = 1.0

        self.register_buffer("cdf", cdf)

    def forward(self, x):

        # -------------------------------------------------
        # First squash to [0,1]
        # -------------------------------------------------

        x = torch.sigmoid(x)

        # -------------------------------------------------
        # Lookup table interpolation
        # -------------------------------------------------

        idx = x * (self.resolution - 1)

        idx0 = torch.floor(idx).long()
        idx1 = torch.clamp(idx0 + 1,
                           max=self.resolution - 1)

        frac = idx - idx0.float()

        y0 = self.cdf[idx0]
        y1 = self.cdf[idx1]

        # Linear interpolation
        out = y0 + frac * (y1 - y0)

        return out


# =========================================================
# SinuNet MLP
# =========================================================

class SinuNet(nn.Module):

    def __init__(self,
                 s=1.0,
                 k=2.0,
                 hidden1=256,
                 hidden2=128):

        super().__init__()

        self.act = FastSinCDFActivation(
            s=s,
            k=k,
            resolution=5000
        )

        self.fc1 = nn.Linear(28 * 28, hidden1)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.fc3 = nn.Linear(hidden2, 10)

    def forward(self, x):

        x = x.view(x.size(0), -1)

        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))

        x = self.fc3(x)

        return x

In [29]:
# SinuNet Model 0.5,5
# =========================================================

model = SinuNet(
    s=0.5,
    k=5
).to(device)

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3
)


# =========================================================
# Run
# =========================================================

EPOCHS = 5

for epoch in range(EPOCHS):

    loss = train_epoch()

    acc = evaluate()

    print(
        f"Epoch {epoch+1} | "
        f"Loss = {loss:.4f} | "
        f"Test Accuracy = {100*acc:.2f}%"
    )

Epoch 1 | Loss = 0.5593 | Test Accuracy = 93.73%
Epoch 2 | Loss = 0.1697 | Test Accuracy = 95.90%
Epoch 3 | Loss = 0.1121 | Test Accuracy = 96.86%
Epoch 4 | Loss = 0.0791 | Test Accuracy = 97.21%
Epoch 5 | Loss = 0.0592 | Test Accuracy = 97.40%


In [30]:
# SinuNet Model 1,5
# =========================================================

model = SinuNet(
    s=1,
    k=5
).to(device)

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3
)


# =========================================================
# Run
# =========================================================

EPOCHS = 5

for epoch in range(EPOCHS):

    loss = train_epoch()

    acc = evaluate()

    print(
        f"Epoch {epoch+1} | "
        f"Loss = {loss:.4f} | "
        f"Test Accuracy = {100*acc:.2f}%"
    )

Epoch 1 | Loss = 0.4441 | Test Accuracy = 94.72%
Epoch 2 | Loss = 0.1418 | Test Accuracy = 96.07%
Epoch 3 | Loss = 0.0903 | Test Accuracy = 97.33%
Epoch 4 | Loss = 0.0626 | Test Accuracy = 97.62%
Epoch 5 | Loss = 0.0443 | Test Accuracy = 97.56%


In [31]:
# SinuNet Model 3,5
# =========================================================

model = SinuNet(
    s=3,
    k=5
).to(device)

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3
)


# =========================================================
# Run
# =========================================================

EPOCHS = 5

for epoch in range(EPOCHS):

    loss = train_epoch()

    acc = evaluate()

    print(
        f"Epoch {epoch+1} | "
        f"Loss = {loss:.4f} | "
        f"Test Accuracy = {100*acc:.2f}%"
    )

Epoch 1 | Loss = 0.4462 | Test Accuracy = 95.63%
Epoch 2 | Loss = 0.1238 | Test Accuracy = 96.87%
Epoch 3 | Loss = 0.0764 | Test Accuracy = 97.40%
Epoch 4 | Loss = 0.0541 | Test Accuracy = 97.53%
Epoch 5 | Loss = 0.0385 | Test Accuracy = 97.80%


In [32]:
# SinuNet Model 0.5,3
# =========================================================

model = SinuNet(
    s=0.5,
    k=3
).to(device)

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3
)


# =========================================================
# Run
# =========================================================

EPOCHS = 5

for epoch in range(EPOCHS):

    loss = train_epoch()

    acc = evaluate()

    print(
        f"Epoch {epoch+1} | "
        f"Loss = {loss:.4f} | "
        f"Test Accuracy = {100*acc:.2f}%"
    )

Epoch 1 | Loss = 0.5732 | Test Accuracy = 93.55%
Epoch 2 | Loss = 0.1827 | Test Accuracy = 95.73%
Epoch 3 | Loss = 0.1220 | Test Accuracy = 96.59%
Epoch 4 | Loss = 0.0884 | Test Accuracy = 97.19%
Epoch 5 | Loss = 0.0659 | Test Accuracy = 97.47%


In [33]:
# SinuNet Model 1,3
# =========================================================

model = SinuNet(
    s=1,
    k=3
).to(device)

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3
)


# =========================================================
# Run
# =========================================================

EPOCHS = 5

for epoch in range(EPOCHS):

    loss = train_epoch()

    acc = evaluate()

    print(
        f"Epoch {epoch+1} | "
        f"Loss = {loss:.4f} | "
        f"Test Accuracy = {100*acc:.2f}%"
    )

Epoch 1 | Loss = 0.4863 | Test Accuracy = 94.01%
Epoch 2 | Loss = 0.1596 | Test Accuracy = 96.32%
Epoch 3 | Loss = 0.1042 | Test Accuracy = 96.85%
Epoch 4 | Loss = 0.0731 | Test Accuracy = 97.30%
Epoch 5 | Loss = 0.0543 | Test Accuracy = 97.44%


In [34]:
# SinuNet Model 3,3
# =========================================================

model = SinuNet(
    s=0.5,
    k=3
).to(device)

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3
)


# =========================================================
# Run
# =========================================================

EPOCHS = 5

for epoch in range(EPOCHS):

    loss = train_epoch()

    acc = evaluate()

    print(
        f"Epoch {epoch+1} | "
        f"Loss = {loss:.4f} | "
        f"Test Accuracy = {100*acc:.2f}%"
    )

Epoch 1 | Loss = 0.5715 | Test Accuracy = 93.58%
Epoch 2 | Loss = 0.1797 | Test Accuracy = 95.68%
Epoch 3 | Loss = 0.1203 | Test Accuracy = 96.64%
Epoch 4 | Loss = 0.0871 | Test Accuracy = 97.21%
Epoch 5 | Loss = 0.0654 | Test Accuracy = 97.37%


In [35]:
# SinuNet Model 0.5,1
# =========================================================

model = SinuNet(
    s=0.5,
    k=1
).to(device)

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3
)


# =========================================================
# Run
# =========================================================

EPOCHS = 5

for epoch in range(EPOCHS):

    loss = train_epoch()

    acc = evaluate()

    print(
        f"Epoch {epoch+1} | "
        f"Loss = {loss:.4f} | "
        f"Test Accuracy = {100*acc:.2f}%"
    )

Epoch 1 | Loss = 0.6488 | Test Accuracy = 92.46%
Epoch 2 | Loss = 0.2056 | Test Accuracy = 94.94%
Epoch 3 | Loss = 0.1429 | Test Accuracy = 96.20%
Epoch 4 | Loss = 0.1078 | Test Accuracy = 96.65%
Epoch 5 | Loss = 0.0840 | Test Accuracy = 97.11%


In [36]:
# SinuNet Model 1,1
# =========================================================

model = SinuNet(
    s=1,
    k=1
).to(device)

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3
)


# =========================================================
# Run
# =========================================================

EPOCHS = 5

for epoch in range(EPOCHS):

    loss = train_epoch()

    acc = evaluate()

    print(
        f"Epoch {epoch+1} | "
        f"Loss = {loss:.4f} | "
        f"Test Accuracy = {100*acc:.2f}%"
    )

Epoch 1 | Loss = 0.5806 | Test Accuracy = 92.90%
Epoch 2 | Loss = 0.1911 | Test Accuracy = 95.35%
Epoch 3 | Loss = 0.1322 | Test Accuracy = 96.28%
Epoch 4 | Loss = 0.0977 | Test Accuracy = 96.83%
Epoch 5 | Loss = 0.0755 | Test Accuracy = 97.22%


In [37]:
# SinuNet Model 3,1
# =========================================================

model = SinuNet(
    s=3,
    k=1
).to(device)

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3
)


# =========================================================
# Run
# =========================================================

EPOCHS = 5

for epoch in range(EPOCHS):

    loss = train_epoch()

    acc = evaluate()

    print(
        f"Epoch {epoch+1} | "
        f"Loss = {loss:.4f} | "
        f"Test Accuracy = {100*acc:.2f}%"
    )

Epoch 1 | Loss = 0.5004 | Test Accuracy = 94.18%
Epoch 2 | Loss = 0.1643 | Test Accuracy = 95.91%
Epoch 3 | Loss = 0.1088 | Test Accuracy = 96.75%
Epoch 4 | Loss = 0.0784 | Test Accuracy = 97.40%
Epoch 5 | Loss = 0.0582 | Test Accuracy = 97.62%
